# 第三方数据拉取接口测试：固定 2026-07-03（v2 修复代理超时）

本 notebook 用于单独测试第三方气象接口，不依赖前端、Flask 后端或 SQLite 缓存。

本版修复两个常见问题：

1. Windows/Conda 环境缺少 `tzdata` 导致 `ZoneInfoNotFoundError`：本版不使用 `zoneinfo`。
2. `requests` 自动读取系统代理，导致请求被转到 `127.0.0.1:7897` 并超时：本版默认禁用系统代理。

固定请求时间范围：

```text
2026-07-03 00:00:00 至 2026-07-03 23:59:59
```

如果你确实需要代理，把配置区的 `USE_SYSTEM_PROXY = True`。

In [1]:
import os
import re
import json
import time
import hmac
import hashlib
from pathlib import Path
from datetime import datetime
from urllib.parse import urlencode
from collections import Counter

try:
    import requests
except ImportError as exc:
    raise ImportError("当前环境缺少 requests，请先执行：pip install requests") from exc


def load_env_file(path):
    path = Path(path)
    if not path.exists():
        return {}
    data = {}
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        data[key.strip()] = value.strip().strip('"').strip("'")
    return data


def load_env_candidates():
    candidates = [
        Path(".env"),
        Path(".env.local"),
        Path("backend") / ".env",
        Path("backend") / ".env.local",
        Path.cwd().parent / ".env",
        Path.cwd().parent / "backend" / ".env",
    ]
    loaded = []
    merged = {}
    for p in candidates:
        if p.exists():
            env = load_env_file(p)
            merged.update(env)
            loaded.append(str(p.resolve()))
    for k, v in merged.items():
        os.environ[k] = v
    return loaded, merged


loaded_env_files, merged_env = load_env_candidates()
print("loaded .env files:")
for item in loaded_env_files:
    print("  -", item)
if not loaded_env_files:
    print("  未找到 .env，将只使用当前系统环境变量。")

loaded .env files:
  - E:\OneDrive\研究生文件\课题组资料\EatRice团队\横向课题\雷祖祥老师\code\atmosphere-borad\backend\.env


In [2]:
# =========================
# 配置区
# =========================

WEATHER_API_BASE_URL = os.getenv("WEATHER_API_BASE_URL", "http://weather-api.jsjldzkj.com/api").rstrip("/")
WEATHER_APP_ID = os.getenv("WEATHER_APP_ID", "dashboard")
WEATHER_SECRET_KEY = (
    os.getenv("WEATHER_SECRET_KEY")
    or os.getenv("WEATHER_SECRET")
    or os.getenv("SECRET_KEY")
    or ""
)

# 修复点 1：这里固定 60 秒，不再被 .env 里的 WEATHER_TIMEOUT_SECONDS=10 覆盖。
TIMEOUT_SECONDS = 60

# 修复点 2：默认禁用系统代理，避免 requests 自动走 127.0.0.1:7897。
# 如果你的网络必须使用 Clash/代理，把它改成 True。
USE_SYSTEM_PROXY = False

# 默认测试原版兼容模式。
DEFAULT_TIME_FORMAT = "ms"       # ms / text
DEFAULT_SIGN_MODE = "json"       # json / concat
DEFAULT_PAGE_SIZE = 500

print("WEATHER_API_BASE_URL:", WEATHER_API_BASE_URL)
print("WEATHER_APP_ID      :", WEATHER_APP_ID)
print("has secret key      :", bool(WEATHER_SECRET_KEY))
print("timeout seconds     :", TIMEOUT_SECONDS)
print("USE_SYSTEM_PROXY    :", USE_SYSTEM_PROXY)

if not WEATHER_SECRET_KEY:
    print("\nWARNING: WEATHER_SECRET_KEY 为空。请检查 .env 或在本 cell 中手动设置 WEATHER_SECRET_KEY。")

WEATHER_API_BASE_URL: http://weather-api.jsjldzkj.com/api
WEATHER_APP_ID      : dashboard
has secret key      : True
timeout seconds     : 60
USE_SYSTEM_PROXY    : False


In [3]:
# =========================
# 固定时间：2026-07-03
# =========================

start_dt = datetime(2026, 7, 3, 0, 0, 0)
end_dt = datetime(2026, 7, 3, 23, 59, 59)

def to_ms(dt):
    # 按本机本地时区解释 naive datetime，生成 13 位毫秒时间戳。
    return int(dt.timestamp() * 1000)

def format_time_pair(time_format="ms"):
    if time_format == "ms":
        return str(to_ms(start_dt)), str(to_ms(end_dt))
    if time_format == "text":
        return start_dt.strftime("%Y-%m-%d %H:%M:%S"), end_dt.strftime("%Y-%m-%d %H:%M:%S")
    raise ValueError("time_format must be 'ms' or 'text'")

print("fixed date range:")
print("start_dt:", start_dt)
print("end_dt  :", end_dt)
print("ms      :", format_time_pair("ms"))
print("text    :", format_time_pair("text"))

fixed date range:
start_dt: 2026-07-03 00:00:00
end_dt  : 2026-07-03 23:59:59
ms      : ('1783008000000', '1783094399000')
text    : ('2026-07-03 00:00:00', '2026-07-03 23:59:59')


In [4]:
# =========================
# 禁用/启用代理的 Session
# =========================

def make_session(use_system_proxy=False):
    session = requests.Session()
    # trust_env=False 会让 requests 忽略 HTTP_PROXY/HTTPS_PROXY/ALL_PROXY 等环境代理。
    # 你截图中的 127.0.0.1:7897 超时就是典型系统代理被自动使用。
    session.trust_env = bool(use_system_proxy)
    return session

session = make_session(USE_SYSTEM_PROXY)

print("session.trust_env:", session.trust_env)
print("environment proxy variables:")
for key in ["HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"]:
    value = os.environ.get(key)
    if value:
        print(f"  {key}={value}")
print("当前默认请求将{}系统代理。".format("使用" if USE_SYSTEM_PROXY else "不使用"))

session.trust_env: False
environment proxy variables:
当前默认请求将不使用系统代理。


In [5]:
# =========================
# 签名与请求函数
# =========================

def make_timestamp_ms():
    return str(int(time.time() * 1000))


def make_signature(appid, timestamp, secret_key, sign_mode="json"):
    if sign_mode == "json":
        plain = json.dumps({"appid": appid, "timestamp": timestamp}, separators=(",", ":"), ensure_ascii=False)
    elif sign_mode == "concat":
        plain = f"{appid}{timestamp}"
    else:
        raise ValueError("sign_mode must be 'json' or 'concat'")
    digest = hmac.new(secret_key.encode("utf-8"), plain.encode("utf-8"), hashlib.sha256).hexdigest()
    return digest, plain


def extract_records(payload):
    if not isinstance(payload, dict):
        return []
    result = payload.get("result")
    if isinstance(result, dict):
        for key in ("list", "data", "records", "rows"):
            value = result.get(key)
            if isinstance(value, list):
                return value
    for key in ("list", "data", "records", "rows"):
        value = payload.get(key)
        if isinstance(value, list):
            return value
    return []


def request_device_data(page=1, page_size=500, time_format="ms", sign_mode="json", verbose=True, use_system_proxy=None):
    if use_system_proxy is None:
        use_system_proxy = USE_SYSTEM_PROXY

    local_session = make_session(use_system_proxy)
    timestamp = make_timestamp_ms()
    authorization, sign_plain = make_signature(WEATHER_APP_ID, timestamp, WEATHER_SECRET_KEY, sign_mode=sign_mode)

    start_value, end_value = format_time_pair(time_format=time_format)
    url = f"{WEATHER_API_BASE_URL}/getDeviceData/{page}/{page_size}"

    headers = {
        "Authorization": authorization,
        "timestamp": timestamp,
    }
    params = {
        "search[start_time]": start_value,
        "search[end_time]": end_value,
    }

    if verbose:
        print("=" * 88)
        print("request")
        print("url              :", url)
        print("page             :", page)
        print("page_size        :", page_size)
        print("time_format      :", time_format)
        print("sign_mode        :", sign_mode)
        print("use_system_proxy :", use_system_proxy)
        print("timeout_seconds  :", TIMEOUT_SECONDS)
        print("sign_plain       :", sign_plain)
        print("timestamp        :", timestamp)
        print("params           :", params)
        print("curl:")
        print(
            "curl "
            + "-H " + repr(f"Authorization: {authorization}") + " "
            + "-H " + repr(f"timestamp: {timestamp}") + " "
            + repr(url + "?" + urlencode(params))
        )

    t0 = time.time()
    response = local_session.get(url, headers=headers, params=params, timeout=TIMEOUT_SECONDS)
    elapsed = time.time() - t0

    text = response.text
    try:
        payload = response.json()
    except Exception:
        payload = None

    records = extract_records(payload) if payload is not None else []

    if verbose:
        print("-" * 88)
        print("response")
        print("status_code  :", response.status_code)
        print("elapsed      :", f"{elapsed:.3f}s")
        print("content_type :", response.headers.get("content-type"))
        print("text preview :")
        print(text[:2000])
        print("-" * 88)
        print("parsed")
        print("json parsed  :", payload is not None)
        print("records count:", len(records))
        if records[:1]:
            print("first record :")
            print(json.dumps(records[0], ensure_ascii=False, indent=2)[:2000])

    return {
        "url": url,
        "headers": headers,
        "params": params,
        "status_code": response.status_code,
        "elapsed": elapsed,
        "text": text,
        "payload": payload,
        "records": records,
        "time_format": time_format,
        "sign_mode": sign_mode,
        "sign_plain": sign_plain,
        "used_system_proxy": use_system_proxy,
    }

In [6]:
# =========================
# 单次请求：默认禁用代理
# =========================
# 如果这里仍然超时，先看报错里 host 是否还是 127.0.0.1:7897。
# 如果不是 127.0.0.1:7897，而是 weather-api.jsjldzkj.com 超时，说明第三方接口本身慢或网络不可达。

try:
    result = request_device_data(
        page=1,
        page_size=DEFAULT_PAGE_SIZE,
        time_format=DEFAULT_TIME_FORMAT,
        sign_mode=DEFAULT_SIGN_MODE,
        verbose=True,
        use_system_proxy=False,
    )
except requests.exceptions.ProxyError as exc:
    print("代理错误：", repr(exc))
    print("当前请求已设置 use_system_proxy=False。请检查是否有系统级透明代理或安全软件拦截。")
    raise
except requests.exceptions.ReadTimeout as exc:
    print("读取超时：", repr(exc))
    print("若报错 host=127.0.0.1, port=7897，说明仍在走代理；请确认 USE_SYSTEM_PROXY=False 后重新运行前面的 cell。")
    print("若报错 host=weather-api.jsjldzkj.com，则说明第三方接口或网络链路超时。")
    raise

request
url              : http://weather-api.jsjldzkj.com/api/getDeviceData/1/500
page             : 1
page_size        : 500
time_format      : ms
sign_mode        : json
use_system_proxy : False
timeout_seconds  : 60
sign_plain       : {"appid":"dashboard","timestamp":"1783307037477"}
timestamp        : 1783307037477
params           : {'search[start_time]': '1783008000000', 'search[end_time]': '1783094399000'}
curl:
curl -H 'Authorization: a1571f2899652a4565dbe324bd0edc10b4bc3c81a643656bd64fa85157fe9874' -H 'timestamp: 1783307037477' 'http://weather-api.jsjldzkj.com/api/getDeviceData/1/500?search%5Bstart_time%5D=1783008000000&search%5Bend_time%5D=1783094399000'
----------------------------------------------------------------------------------------
response
status_code  : 200
elapsed      : 39.605s
content_type : application/json; charset=utf-8
text preview :
{"code":0,"message":"common.success","result":{"total":1408,"list":[{"id":774783,"serial_number":"","jiebin":"0","shuimo":"0

In [ ]:
# =========================
# 可选：使用系统代理再测一次
# =========================
# 只有当你的网络必须通过 Clash/代理访问外网时才运行。
# 如果代理端口 127.0.0.1:7897 不稳定，这个 cell 可能继续超时。

# result_proxy = request_device_data(
#     page=1,
#     page_size=DEFAULT_PAGE_SIZE,
#     time_format=DEFAULT_TIME_FORMAT,
#     sign_mode=DEFAULT_SIGN_MODE,
#     verbose=True,
#     use_system_proxy=True,
# )

In [ ]:
# =========================
# 组合测试：ms/text × json/concat
# =========================
# 如果单次请求成功但记录为 0，可运行这个 cell 检查时间格式和签名模式。

matrix_results = []

for time_format in ["ms", "text"]:
    for sign_mode in ["json", "concat"]:
        try:
            r = request_device_data(
                page=1,
                page_size=DEFAULT_PAGE_SIZE,
                time_format=time_format,
                sign_mode=sign_mode,
                verbose=False,
                use_system_proxy=False,
            )
            item = {
                "time_format": time_format,
                "sign_mode": sign_mode,
                "status_code": r["status_code"],
                "elapsed": round(r["elapsed"], 3),
                "records": len(r["records"]),
                "text_preview": r["text"][:300].replace("\n", " "),
            }
            print(f"{time_format:4s} | {sign_mode:6s} | status={r['status_code']} | records={len(r['records']):5d} | elapsed={r['elapsed']:.3f}s")
        except Exception as exc:
            item = {
                "time_format": time_format,
                "sign_mode": sign_mode,
                "error": repr(exc),
            }
            print(f"{time_format:4s} | {sign_mode:6s} | ERROR: {exc!r}")
        matrix_results.append(item)

print("\nsummary:")
print(json.dumps(matrix_results, ensure_ascii=False, indent=2)[:5000])

In [7]:
# =========================
# 分页拉取 2026-07-03 全天数据
# =========================

def extract_record_time(record):
    if not isinstance(record, dict):
        return None

    candidate_keys = [
        "time", "createTime", "create_time", "created_at", "createdAt",
        "recordTime", "record_time", "dataTime", "data_time",
        "collectTime", "collect_time", "timestamp",
    ]

    value = None
    for key in candidate_keys:
        if key in record and record[key] not in (None, ""):
            value = record[key]
            break

    if value is None:
        return None

    if isinstance(value, (int, float)):
        v = float(value)
        if v > 10_000_000_000:
            v = v / 1000.0
        try:
            return datetime.fromtimestamp(v)
        except Exception:
            return None

    text = str(value).strip()
    if not text:
        return None

    if re.fullmatch(r"\d{13}", text):
        try:
            return datetime.fromtimestamp(int(text) / 1000.0)
        except Exception:
            return None

    if re.fullmatch(r"\d{10}", text):
        try:
            return datetime.fromtimestamp(int(text))
        except Exception:
            return None

    for fmt in [
        "%Y-%m-%d %H:%M:%S",
        "%Y/%m/%d %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%dT%H:%M:%S.%f",
    ]:
        try:
            return datetime.strptime(text[:26], fmt)
        except Exception:
            pass

    try:
        return datetime.fromisoformat(text.replace("Z", "").replace("+08:00", ""))
    except Exception:
        return None


def fetch_all_pages_for_20260703(page_size=500, max_pages=160, time_format="ms", sign_mode="json", sleep_seconds=0.05):
    all_records = []
    page_summaries = []

    for page in range(1, max_pages + 1):
        print(f"requesting page {page} ...")
        try:
            r = request_device_data(
                page=page,
                page_size=page_size,
                time_format=time_format,
                sign_mode=sign_mode,
                verbose=False,
                use_system_proxy=False,
            )
        except Exception as exc:
            print(f"  stop: exception on page {page}: {exc!r}")
            break

        records = r["records"]
        all_records.extend(records)

        page_summaries.append({
            "page": page,
            "status_code": r["status_code"],
            "records": len(records),
            "elapsed": round(r["elapsed"], 3),
        })

        print(f"  page={page}; status={r['status_code']}; records={len(records)}; total_so_far={len(all_records)}")

        if r["status_code"] != 200:
            print("  stop: non-200 response")
            break
        if not records:
            print("  stop: empty page")
            break
        if len(records) < page_size:
            print("  stop: short page")
            break

        time.sleep(sleep_seconds)

    return all_records, page_summaries


all_records_20260703, page_summaries = fetch_all_pages_for_20260703(
    page_size=DEFAULT_PAGE_SIZE,
    max_pages=160,
    time_format=DEFAULT_TIME_FORMAT,
    sign_mode=DEFAULT_SIGN_MODE,
)

print("\npage summaries:")
print(json.dumps(page_summaries, ensure_ascii=False, indent=2)[:5000])
print("total records:", len(all_records_20260703))

requesting page 1 ...
  page=1; status=200; records=500; total_so_far=500
requesting page 2 ...
  page=2; status=200; records=500; total_so_far=1000
requesting page 3 ...
  page=3; status=200; records=408; total_so_far=1408
  stop: short page

page summaries:
[
  {
    "page": 1,
    "status_code": 200,
    "records": 500,
    "elapsed": 22.981
  },
  {
    "page": 2,
    "status_code": 200,
    "records": 500,
    "elapsed": 10.871
  },
  {
    "page": 3,
    "status_code": 200,
    "records": 408,
    "elapsed": 22.612
  }
]
total records: 1408


In [ ]:
with open("")

[{'id': 774783,
  'serial_number': '',
  'jiebin': '0',
  'shuimo': '0',
  'jixue': '0',
  'wendu': '25.5',
  'shidu': '95.3',
  'nengjiandu': '1640',
  'guangqiang': '0',
  'fengsu': '8.5',
  'dianliu': '16',
  'fengxiang': '269.2',
  'pm25': '26.8',
  'pm10': '26.6',
  'yuliang': '162.13',
  'yuqiang': '0',
  'guangdian': '27525',
  'fengdian': '67601',
  'yadian': '7864',
  'dianya': '1222',
  'correct': '0',
  'receive_time': '2026-07-03 23:59:37',
  'create_time': '2026-07-03 23:59:37',
  'update_time': '2026-07-03 23:59:37',
  'delete_time': 0},
 {'id': 774782,
  'serial_number': '',
  'jiebin': '0',
  'shuimo': '0',
  'jixue': '0',
  'wendu': '25.6',
  'shidu': '95.5',
  'nengjiandu': '1660',
  'guangqiang': '0',
  'fengsu': '8.5',
  'dianliu': '14',
  'fengxiang': '269.2',
  'pm25': '32.9',
  'pm10': '37.2',
  'yuliang': '162.13',
  'yuqiang': '0',
  'guangdian': '27525',
  'fengdian': '55786',
  'yadian': '7864',
  'dianya': '1222',
  'correct': '0',
  'receive_time': '2026-07

In [ ]:
# =========================
# 每小时记录数与样例记录
# =========================

hour_counter = Counter()
unknown_time = 0

for rec in all_records_20260703:
    dt = extract_record_time(rec)
    if dt is None:
        unknown_time += 1
        continue
    if start_dt <= dt <= end_dt:
        hour_counter[dt.hour] += 1

print("hourly counts for 2026-07-03:")
for hour in range(24):
    print(f"{hour:02d}:00 - {hour:02d}:59  {hour_counter.get(hour, 0)}")

print("unknown_time records:", unknown_time)

print("\nfirst 3 records:")
for i, rec in enumerate(all_records_20260703[:3], 1):
    print(f"\n--- record {i} ---")
    print(json.dumps(rec, ensure_ascii=False, indent=2)[:3000])